# PatchCore anomalib backend — Colab driver

Phases 0–2 of `docs/superpowers/plans/2026-07-24-patchcore-anomalib-backend-colab.md`.
**The playbook is the authority**; this notebook is its runnable form. Phases 3 (the VisA ±1pt
gate) and 4 (freeze provenance) stay in the playbook because they need a loader and a published
table that cannot be pre-written — see the last section.

**Runtime → Change runtime type → T4 GPU** before running anything.

## What this session must prove, in order

| Phase | Proves | Gate |
|---|---|---|
| 0 | anomalib version + API shape | if 0.3 mismatches, **stop and adapt Phase 1** |
| 1 | the backend works | the smoke test, not the docs |
| 2 | it drives the whole runner path | I-AUROC well above 0.5 on `regular` |

**Honest framing, from the playbook:** every anomalib call here comes from the library's published
API, but the exact `>=1.1` signatures were never executed in the environment that wrote it. Step 0.3
and the Phase 1 smoke test are the real verification. **A mismatch there is expected maintenance,
not a plan failure** — adapt the call to the installed version and continue. The one thing that is
never adapted away is the ±1.0 VisA gate.

## Phase 0 — environment and API verification

### 0.1 GPU + repo

In [ ]:
!nvidia-smi -L      # must print a GPU; if not: Runtime -> Change runtime type -> T4 GPU

REPO = "/content/vlm-anomaly-bench"
!git clone https://github.com/andrudebaran7/vlm-anomaly-bench.git {REPO}
%cd {REPO}

# NOT --quiet: a failed editable install here surfaces four cells later as a baffling
# "No module named 'vlmab'", and the quiet flag is what hides the real reason.
!pip install -e .

# Belt and braces: the repo uses a src/ layout, so if a later dependency install disturbs
# the editable-install finder, this keeps vlmab importable anyway.
import sys
if REPO + "/src" not in sys.path:
    sys.path.insert(0, REPO + "/src")

import vlmab
print("vlmab imports OK from", vlmab.__file__)

### 0.2 Install anomalib and record the resolved version

The version this prints is provenance: it goes into `configs/methods/patchcore_ref.yaml` in Phase 4,
recorded as the version the VisA gate passed on.

In [ ]:
!pip install "anomalib==2.6.0"

import anomalib, torch
ANOMALIB_VERSION = anomalib.__version__
print("anomalib", ANOMALIB_VERSION, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "no GPU - the backend cannot run"

# Pinned, not a range. configs/methods/patchcore_ref.yaml freezes 2.6.0 as the version every
# measured number in this repo came from, and the five ways 2.6.0 contradicts its own
# documentation (see the backend's comments) are specific to it. A range here would let a fresh
# runtime resolve to something none of those findings apply to, while the config still claimed
# 2.6.0. tests/test_notebook_pins.py holds this line and that file together.
import yaml
PINNED = yaml.safe_load(open("configs/methods/patchcore_ref.yaml"))["anomalib_version"]
assert ANOMALIB_VERSION == PINNED, (
    f"installed anomalib {ANOMALIB_VERSION} but the config freezes {PINNED} - stop here. "
    "Either the pin moved deliberately (then update the config, the backend comments and the "
    "session notes together) or pip resolved something else."
)
print(f"OK: installed anomalib matches the frozen pin ({PINNED})")


### ⚠️ If Colab offered to restart the runtime — read this

Installing anomalib pulls a large dependency tree that often conflicts with Colab's preinstalled
numpy/torch, so Colab commonly prompts **"Restart runtime"**. Accepting is usually the right call,
but a restart throws away:

- the `%cd` from 0.1, so the working directory reverts to `/content`;
- every Python variable, including `ANOMALIB_VERSION`;
- sometimes the editable install's path hook.

Files on disk survive — the cloned repo and anything `%%writefile` already wrote are still there.

**Run the cell below after any restart.** It is idempotent, so running it when you did not restart
costs nothing.

In [ ]:
# Idempotent restart recovery: safe to run at any point, any number of times.
%cd /content/vlm-anomaly-bench

import sys
if "/content/vlm-anomaly-bench/src" not in sys.path:
    sys.path.insert(0, "/content/vlm-anomaly-bench/src")

import vlmab, anomalib, torch
ANOMALIB_VERSION = anomalib.__version__          # restore the provenance string
print("vlmab OK | anomalib", ANOMALIB_VERSION,
      "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

import os
print("backend file written:",
      os.path.isfile("src/vlmab/methods/patchcore_backend.py"),
      "(True -- it is a tracked file, so the clone already carries it)")

### 0.3 VERIFY the API against the installed version

**This is the cheapest place to catch API drift, and the single most likely failure of the plan.**
Read each output and confirm it matches what Phase 1 assumes. Note any mismatch — you adapt Phase 1
to the installed signature, never the other way round.

In [ ]:
import inspect
from anomalib.models import Patchcore
from anomalib.engine import Engine
from anomalib.data import Folder

# (a) constructor takes the pre-registered hyperparameters?
#     expect: backbone, layers, coreset_sampling_ratio, num_neighbors
print("(a) Patchcore.__init__:\n", inspect.signature(Patchcore.__init__), "\n")

# (b) the default pre-processor transform score() will use?
#     expect: Resize([256, 256]) -> Normalize(imagenet), with NO CenterCrop.
#     This is the corrected expectation. The original said "Resize(256) -> CenterCrop(224) ->
#     Normalize" -- what classic PatchCore does -- and the 2026-08-21 session DISPROVED it.
#     That difference is a 32x32 patch grid instead of 28x28, i.e. 31% more patches over the
#     full frame instead of the centre crop, and it is the measured suspect for the gate
#     failing by 0.057 on 2026-09-16. Phase 3b runs the other transform.
print("(b) pre-processor transform:\n", Patchcore.configure_pre_processor().transform, "\n")

# (c) Folder accepts normal-only with splits disabled?
#     expect: name, root, normal_dir, val_split_mode, test_split_mode -- and NO `task`
#     (removed in 2.x, though several doc snippets still pass it).
print("(c) Folder.__init__:\n", inspect.signature(Folder.__init__), "\n")

# (d) Engine kwargs the backend passes?
#     expect: (callbacks, logger, default_root_dir, **kwargs) -- everything else reaches
#     Lightning's Trainer lazily, which is why a bad kwarg here dies one call later.
print("(d) Engine.__init__:\n", inspect.signature(Engine.__init__))


**Record your findings before moving on.** (d) is a convenience setting — a rejected kwarg gets
dropped. (a) and (b) are correctness: a changed hyperparameter name or a different transform
changes the numbers.

| Check | Matched? | If not, what differs |
|---|---|---|
| (a) constructor | | |
| (b) transform | | |
| (c) Folder | | |
| (d) Engine | | |

### 1.1 Sync the backend from the repo

`src/vlmab/methods/patchcore_backend.py` is a **tracked file in the repo**, not something this
notebook writes. That matters for iteration: when a fix is pushed to `master`, you pick it up by
**re-running this cell** — no editing, no pasting, no reloading the notebook.

It also puts the source where source belongs. Code that lives only inside a notebook's JSON cannot
be diffed, reviewed, or imported by anything else.

In [ ]:
REPO = "/content/vlm-anomaly-bench"

# Hard sync. The clone is scratch, so local edits are never worth keeping -- and a plain
# `git pull` REFUSES when an untracked file collides with an incoming tracked one, which is
# exactly what happens if an older version of this notebook wrote patchcore_backend.py via
# %%writefile. The pull then errors, the cell carries on, and you import stale code while a
# print claims it synced. Reset instead, and verify.
!cd {REPO} && git fetch --quiet origin && git reset --hard --quiet origin/master && git clean -qfd src
!cd {REPO} && git log --oneline -1 && git status --porcelain | head

import sys, importlib, subprocess
for p in (f"{REPO}/src", f"{REPO}/notebooks"):
    if p not in sys.path:
        sys.path.insert(0, p)

# Verify the working tree really matches origin/master before trusting anything it contains.
dirty = subprocess.run(["git", "-C", REPO, "status", "--porcelain"],
                       capture_output=True, text=True).stdout.strip()
assert not dirty, f"working tree is not clean after reset:\n{dirty}"

import vlmab.methods.patchcore_backend as _bk
importlib.reload(_bk)
PatchCoreBackend = _bk.PatchCoreBackend

import inspect
src = inspect.getsource(_bk.PatchCoreBackend.__init__)
print(src)
print("--- the source above is what will run, and it matches origin/master ---")

### 1.1b Probe the backend stage by stage — run this when something breaks

`fit()` does five things behind one call, so a failure lands six frames deep in Lightning and says
nothing about which of *our* assumptions broke. `probe_patchcore` splits it into ten stages, each
declaring what it expects **before** running the smallest thing that tests it, stopping at the first
failure with the state that matters dumped.

Stage 1 is the one that earns its keep: it enumerates what `Folder`/`Engine`/`Patchcore` actually
accept and flags any `**kwargs` sink — a sink accepts an unknown argument at construction and defers
the failure to whoever consumes it, which is why `Engine(task=...)` built fine and then died inside
`fit()`.

The probe does **not** use `PatchCoreBackend`; it rebuilds each step independently. So it works
regardless of what state the backend file is in.

In [ ]:
import probe_patchcore, importlib
importlib.reload(probe_patchcore)

ctx = probe_patchcore.run()              # stops at the first failure
# ctx = probe_patchcore.run(keep_going=True)   # or: run every stage for the full picture

### 1.1c Stage 7 candidates — run this if the probe stops at `engine.train`

**What the probe found (2026-08-21, anomalib 2.6.0):**

```
AttributeError: 'Folder' object has no attribute 'val_data'
```

`val_split_mode=ValSplitMode.NONE` means the datamodule never builds `val_data`, but Lightning's
fit loop sets up the validation loop unconditionally before the first epoch. The two options are
incompatible, and no constructor argument fixes that.

Worth noticing what stage 7's own log says: *"configure_optimizers returned None, this fit will run
with no optimizer"*. PatchCore does not train — it runs a forward pass over the normal images and
subsamples a coreset. The Trainer is scaffolding around that.

`diagnose_train()` tests three candidates on a fresh model each, and reports which fill the memory
bank:

- **A** — keep every training image, tell Lightning not to validate (`limit_val_batches=0`).
  Preferred: PatchCore's memory bank *is* the model, so holding images back changes the result.
- **B** — give the datamodule a real validation split. Works, but costs training images.
- **C** — call the fit path only, skipping `Engine.train`'s test phase.

In [ ]:
import probe_patchcore, importlib
importlib.reload(probe_patchcore)

results = probe_patchcore.diagnose_train()

### 1.2 Smoke test — the real verification

Twenty flat-ish "normal" images and one with a bright injected patch. **The defect must score
higher.** If it does not, the fit or score path is wrong — debug here, on synthetic data, before
touching the real dataset.

The two calls the docs are least certain about are exercised for the first time here: `self._model(tensor)`
returning `.pred_score`/`.anomaly_map`, and the `Engine(...)` kwargs. If the model returns a
different container, adapt only the two extraction lines in `score()`.

> **`ModuleNotFoundError: No module named 'vlmab'` here** means the runtime restarted after the
> anomalib install (or the editable install failed). Run the restart-recovery cell in Phase 0, then
> re-run 1.1 if `patchcore_backend.py` is missing, then come back.

In [ ]:
import numpy as np
from vlmab.methods.patchcore_backend import PatchCoreBackend

rng = np.random.default_rng(0)
normal = [rng.integers(90, 110, (256, 256, 3), dtype=np.uint8) for _ in range(20)]  # flat-ish
anom = normal[0].copy(); anom[40:80, 40:80] = 255                                   # bright patch

b = PatchCoreBackend()
b.fit(iter(normal))
s_normal, m_normal = b.score(normal[1])
s_anom, m_anom = b.score(anom)

assert m_anom.ndim == 2 and np.isfinite(m_anom).all(), (m_anom.shape,)
assert isinstance(s_anom, float) and np.isfinite(s_anom)
assert s_anom > s_normal, (s_anom, s_normal)   # the defect must score higher
print("OK  normal:", round(s_normal, 4), " anomalous:", round(s_anom, 4), " map:", m_anom.shape)

### 1.3 Commit the backend

Set your git identity and a token first (fine-grained, `contents: write` on this repo). If you would
rather not paste a token into Colab, skip every commit cell and download the files at the end
instead.

In [ ]:
import os
from getpass import getpass

os.environ["GH_TOKEN"] = getpass("GitHub token (input hidden): ")
!git config user.email "sierprinsky@gmail.com"
!git config user.name "andrudebaran7"
!git remote set-url origin https://$GH_TOKEN@github.com/andrudebaran7/vlm-anomaly-bench.git
print("remote configured")

In [ ]:
!git add src/vlmab/methods/patchcore_backend.py
!git commit -m "feat: anomalib PatchCore backend bridging the fit/score seam (GPU-only)"
!git push

## Phase 2 — end to end through the existing runner, on Vial

Proves the backend drives the same path `intensity_baseline` already exercises —
`run_evaluation` → shard → `aggregate` — before spending a full grid.

### 2.1 Fetch one category and verify its layout

**Before running the next cell:** upload `vial.tar.gz` — the unmodified archive from the MVTec
download page — to your Drive at `MyDrive/mvtec_ad2/vial.tar.gz`. It is done once; every later
session and every M3 category reuses the same cell by changing `CATEGORY`.

Drive rather than a re-download because the archives are behind mvtec.com's registration form,
which a session cannot clear unattended. 0.77 GB for Vial; see `docs/datasets-access.md` for the
per-category sizes. **Never into `/tmp`** — it is tmpfs, i.e. RAM.

In [ ]:
# --- Fetch one category from Drive into data/mvtec_ad2/<category> ---
# Drive, not a re-download: the category archives sit behind mvtec.com's registration form, so
# a session cannot fetch them unattended. Upload each <category>.tar.gz to Drive ONCE, under
# MyDrive/mvtec_ad2/, and every future session (and every category in M3) reuses this cell.
# Never extract into /tmp -- it is tmpfs, i.e. RAM.
import os, subprocess
from google.colab import drive

CATEGORY = "vial"
drive.mount("/content/drive")

TAR = f"/content/drive/MyDrive/mvtec_ad2/{CATEGORY}.tar.gz"
DEST = "/content/vlm-anomaly-bench/data/mvtec_ad2"

assert os.path.isfile(TAR), (
    f"{TAR} not found. Upload {CATEGORY}.tar.gz to MyDrive/mvtec_ad2/ and re-run this cell. "
    "The archive is the one from the MVTec download page, unmodified."
)

# Idempotent: a re-run after a disconnect must not spend minutes re-extracting 775 MB.
if os.path.isdir(f"{DEST}/{CATEGORY}/train"):
    print(f"{CATEGORY} already extracted at {DEST}/{CATEGORY} -- skipping")
else:
    os.makedirs(DEST, exist_ok=True)
    # The archive's top level is `<category>/`, `license.txt`, `readme.txt` (verified against
    # vial.tar.gz), so -C lands it at <DEST>/<category> with no path surgery.
    # -xf, not -xzf: MVTec AD 2 ships .tar.gz but MVTec AD classic ships .tar.xz, and a
    # hard-coded -z refuses the latter. tar detects the compression either way.
    subprocess.run(["tar", "-xf", TAR, "-C", DEST], check=True)
    print("extracted to", f"{DEST}/{CATEGORY}")

!python scripts/prepare_data.py --root data/mvtec_ad2 --category {CATEGORY}
# Expected: exit 0, "OK", and the verified counts:
#   train regular=291, validation regular=41, test_public 140 across 7 conditions,
#   test_private 276, test_private_mixed 276

### 2.2 Run PatchCore over Vial's public test split

The runner calls `method.fit(train_images, "vial")` from the `train` split (protocol §3) before
scoring the 140 public-test images.

**Watch for the runner's float16 overflow guard.** If it fires, PatchCore's raw scores exceed 65504
— record the value and stop. That means the map needs a documented scale, which is a §4 protocol
amendment, not a silent change.

In [ ]:
from vlmab.datasets.mvtec_ad2 import MVTecAD2
from vlmab.eval.runner import run_evaluation
from vlmab.eval.store import ResultStore
from vlmab.eval.provenance import run_meta
from vlmab.methods.patchcore_ref import PatchCoreRef
from vlmab.methods.patchcore_backend import PatchCoreBackend

# The seed goes on the BACKEND, which is what calls seed_everything. PatchCoreRef reads it
# from there and the runner stamps it into the shard, so the record cannot disagree with the
# run. run_meta no longer takes a seed for exactly that reason.
method = PatchCoreRef(backend=PatchCoreBackend(seed=0))
store = ResultStore("results/patchcore/vial/shards")
meta = run_meta({"method": "patchcore_ref", "split": "test_public"})

run_evaluation(
    MVTecAD2("data/mvtec_ad2"), method, store, meta,
    categories=["vial"], split="test_public",
    maps_dir="results/patchcore/vial/maps",
)
print("declared seed:", method.seed)

### 2.3 Aggregate per lighting condition and sanity-check

Per lighting condition, not per category — that is what MVTec AD 2 exists to measure, and what keeps
native-resolution evaluation inside the memory budget.

**Expected:** I-AUROC **well above 0.5** on `regular`. A full-shot anchor that cannot beat chance on
its easiest condition is broken. A large drop from `regular` to the `shift_*` / `over/underexposed`
conditions is the lighting-robustness story, not an error — record the table either way.

In [ ]:
from vlmab.eval.store import ResultStore
from vlmab.eval.aggregate import aggregate

df = ResultStore("results/patchcore/vial/shards").load_all()
print(aggregate(df, by="meta_lighting")[
    ["meta_lighting", "i_auroc", "au_pro_030", "au_pro_005", "n"]].to_string(index=False))

## Phase 3 — the reproduction gate (the acceptance criterion for M2)

**No MVTec AD 2 number is reported until this passes** (protocol §2).

The gate is **MVTec AD classic at 99.0 ± 1.0 I-AUROC**, not VisA. PatchCore's own paper
(arXiv:2106.08265, v2 May 2022) predates the VisA dataset (July 2022) and reports no VisA
number, so "the number from the method's own paper" is unsatisfiable there. Its MVTec AD
figure is `PatchCore-10` = **99.0**, where the suffix is the coreset ratio — exactly this
repo's `coreset_sampling_ratio: 0.1`. VisA is still run, as a secondary check that reports
with a caveat and cannot fail the method (protocol §2 v0.2.12).

Targets are pre-registered and committed in `configs/reproduction/patchcore_ref.yaml`,
including the 15 per-category values, so the gate cannot be moved after seeing a result.

### 3.1 Fetch all 15 categories from Drive

Upload each `<category>.tar.xz`, unmodified, to `MyDrive/mvtec_ad/`. All 15 are needed: the
published 99.0 is a mean over 15, and the gate refuses an incomplete run by design.

In [ ]:
import os, subprocess
from google.colab import drive

MVTEC_AD_CATEGORIES = [
    "bottle", "cable", "capsule", "carpet", "grid", "hazelnut", "leather", "metal_nut",
    "pill", "screw", "tile", "toothbrush", "transistor", "wood", "zipper",
]
SRC = "/content/drive/MyDrive/mvtec_ad"
DEST = "/content/vlm-anomaly-bench/data/mvtec_ad"

drive.mount("/content/drive")
os.makedirs(DEST, exist_ok=True)

missing = [c for c in MVTEC_AD_CATEGORIES if not os.path.isfile(f"{SRC}/{c}.tar.xz")]
assert not missing, f"not in {SRC}: {missing}. Upload them before running the gate."

for cat in MVTEC_AD_CATEGORIES:
    if os.path.isdir(f"{DEST}/{cat}/train/good"):
        print(f"{cat}: already extracted")
        continue
    # -xf, not -xzf: these are .tar.xz. The archive's top level is <category>/.
    subprocess.run(["tar", "-xf", f"{SRC}/{cat}.tar.xz", "-C", DEST], check=True)
    n = len(os.listdir(f"{DEST}/{cat}/train/good"))
    print(f"{cat}: extracted, {n} train images")

print("\ncategories on disk:", sorted(os.listdir(DEST)))

### 3.2 Run PatchCore over all 15 categories

One category at a time — that is the resume unit, so a disconnect costs one category and not
the run. Re-running this cell skips whatever already has a shard.

MVTec AD classic's test split is called `test` (MVTec AD 2 uses `test_public`), and it has no
`validation` split at all.

**Budget:** Vial's coreset took 2m19s for 291 train images. These categories run 200-400 train
images each, so expect roughly 30-60 minutes for the 15 plus scoring. If the session drops,
re-run — nothing is lost.

In [ ]:
from vlmab.datasets.mvtec_ad import MVTecAD
from vlmab.eval.runner import run_evaluation
from vlmab.eval.store import ResultStore
from vlmab.eval.provenance import run_meta
from vlmab.methods.patchcore_ref import PatchCoreRef
from vlmab.methods.patchcore_backend import PatchCoreBackend

REPRO = "results/reproduction/mvtec_ad"

dataset = MVTecAD("data/mvtec_ad")
store = ResultStore(f"{REPRO}/shards")
meta = run_meta({"method": "patchcore_ref", "split": "test"})

for cat in MVTEC_AD_CATEGORIES:
    # A fresh backend per category: the memory bank is per-category by construction
    # (protocol §3 full-shot anchor), and reusing one would carry the previous bank over.
    method = PatchCoreRef(backend=PatchCoreBackend(seed=0))
    run_evaluation(
        dataset, method, store, meta,
        categories=[cat], split="test", fit_split="train",
        maps_dir=f"{REPRO}/maps",
    )
    print(f"{cat}: done (seed {method.seed})")

### 3.3 Score it against the pre-registered targets

This is CPU work and runs anywhere — it reads the shards and compares. The exit code is 1 on
FAIL, so a red verdict cannot be printed and scrolled past.

**Read the per-category table even on a PASS.** A mean inside ±1.0 with one category seven
points low is not an even reproduction; the report flags that case explicitly.

In [ ]:
!python scripts/reproduction_gate.py \
    --results results/reproduction/mvtec_ad/shards \
    --targets configs/reproduction/patchcore_ref.yaml \
    --out results/reproduction/patchcore_mvtec_ad.md

print(open("results/reproduction/patchcore_mvtec_ad.md").read())

### 3.4 VisA — the secondary check

Reported, never a gate. VisA is on AWS Open Data under CC BY 4.0 with no registration, so it
is fetched straight into the session rather than through Drive (1.8 GB, a couple of minutes).

Its published number is 92.4 (VisA dataset paper, Table 6, 1-class, averaged over 12 objects).
That source states none of its PatchCore hyperparameters and its MVTec-AD control sits above
PatchCore's own published figures, which is exactly why a miss here cannot fail the method.

In [ ]:
import os, subprocess

VISA = "/content/vlm-anomaly-bench/data/visa"
os.makedirs(VISA, exist_ok=True)
if not os.path.isfile(f"{VISA}/split_csv/1cls.csv"):
    subprocess.run(["curl", "-sL", "-o", f"{VISA}/VisA_20220922.tar",
                    "https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar"],
                   check=True)
    size = os.path.getsize(f"{VISA}/VisA_20220922.tar")
    assert size == 1_929_840_640, f"expected 1,929,840,640 bytes, got {size:,}"
    subprocess.run(["tar", "-xf", f"{VISA}/VisA_20220922.tar", "-C", VISA], check=True)
    os.remove(f"{VISA}/VisA_20220922.tar")   # the extracted tree is what is needed

# Imported here rather than relied on from 3.2, so this check can be run on its own.
from vlmab.datasets.visa import VisA
from vlmab.eval.runner import run_evaluation
from vlmab.eval.store import ResultStore
from vlmab.eval.provenance import run_meta
from vlmab.methods.patchcore_ref import PatchCoreRef
from vlmab.methods.patchcore_backend import PatchCoreBackend

VISA_REPRO = "results/reproduction/visa"
visa = VisA(VISA)
visa_store = ResultStore(f"{VISA_REPRO}/shards")
visa_meta = run_meta({"method": "patchcore_ref", "split": "test"})

for cat in visa.categories():
    method = PatchCoreRef(backend=PatchCoreBackend(seed=0))
    run_evaluation(
        visa, method, visa_store, visa_meta,
        categories=[cat], split="test", fit_split="train",
        maps_dir=f"{VISA_REPRO}/maps",
    )
    print(f"{cat}: done")

In [ ]:
!python scripts/reproduction_gate.py \
    --results results/reproduction/visa/shards \
    --targets configs/reproduction/patchcore_ref.yaml \
    --which secondary \
    --out results/reproduction/patchcore_visa.md

print(open("results/reproduction/patchcore_visa.md").read())

## Phase 3b — the CenterCrop investigation (the gate failed by 0.057)

Phase 3 measured **97.94** against a published 99.0: a FAIL by 0.057, recorded in
`results/reproduction/patchcore_mvtec_ad.md`. `toothbrush` alone contributes **-0.591** of the
-1.057 shortfall.

Two causes were pre-registered **before** that run, so acting on either is investigation, not
tuning-after-the-fact (protocol §7):

1. only one seed was run, where §6 requires three, and
2. the **CenterCrop deviation** — anomalib 2.6.0 pre-processes to 256x256 with no crop, a 32x32
   patch grid, where classic PatchCore's `Resize(256) -> CenterCrop(224)` gives 28x28. Measured
   on 2026-09-16: every category yielded exactly 102.4 coreset points per training image, i.e.
   1024 patches, i.e. **31% more patches than the paper's configuration**.

This phase re-runs all 15 categories with `preprocess="classic"` into a **separate results
root**, and scores it. **Both reports are committed, not just a passing one.**

Two things this phase does not do, deliberately:

- It does not touch the phase-3 report. That FAIL stands on the record whatever this produces.
- It does not report a number from one seed. §6 needs three; cell 3b.3 is how to get there once
  this run has shown whether the pre-processing is the cause at all.

### 3b.0 VERIFY the classic pre-processor actually reaches the model

**Run this before anything else. It takes about a minute and it is what stops a 40-minute run
of plausible, wrong numbers.**

anomalib 2.6.0's `PreProcessor` API is not in this repo's verified record — the 2026-08-21
session verified the `Engine`, and this backend has already been bitten five separate times by a
call that looked reasonable and had never been run. So probe stage 13 prints what is really
there, then checks the **consequence** rather than the API: it fits on 20 synthetic images under
both pre-processings and compares the coreset size, which must be 784 patches per image for
`classic` against 1024 for `anomalib`.

If it fails, fix the backend from what it prints — not from the documentation — and do not
continue to 3b.1.

In [ ]:
import probe_patchcore, importlib
importlib.reload(probe_patchcore)

STAGE = "13. the classic pre-processor"
ctx = probe_patchcore.run(only=STAGE)

# Two asserts, not one. `run(only=...)` with a name that matches nothing runs ZERO stages and
# returns no failure — so "did not fail" is not evidence that anything was checked, and on its
# own it would wave a 40-minute fit through unverified. The coreset sizes are the evidence.
assert "coreset_classic" in ctx and "coreset_anomalib" in ctx, (
    f"{STAGE!r} did not run — has it been renamed? Nothing was verified; do not run 3b.1."
)
assert ctx.get("_failed_at") is None, "stage 13 failed — do not run 3b.1 until it passes"
print(f"\nverified: classic {ctx['coreset_classic']} coreset rows vs "
      f"anomalib {ctx['coreset_anomalib']}")


### 3b.1 Run all 15 categories with the classic pre-processor

Same 15 categories, same seed 0, same everything else — one variable changed, into
`results/reproduction/mvtec_ad_classic/`. A separate root is required, not tidiness: the shard
filename carries only dataset/method/category/seed, so two pre-processings in one root would
overwrite each other's shards and share one map directory. The gate refuses a root that pools
them.

**Budget:** the same 30-60 minutes phase 3.2 took. One category at a time is the resume unit, so
a disconnect costs one category.

In [ ]:
from vlmab.datasets.mvtec_ad import MVTecAD
from vlmab.eval.runner import run_evaluation
from vlmab.eval.store import ResultStore
from vlmab.eval.provenance import run_meta
from vlmab.methods.patchcore_ref import PatchCoreRef
from vlmab.methods.patchcore_backend import PatchCoreBackend

PREPROCESS = "classic"
SEED = 0
REPRO_CLASSIC = "results/reproduction/mvtec_ad_classic"

dataset = MVTecAD("data/mvtec_ad")
store = ResultStore(f"{REPRO_CLASSIC}/shards")

# `preprocess` goes into the cfg run_meta hashes, so this run gets its own config_hash and its
# own map directory. Without it the two runs would collide there — run_id keys map directories
# on config_hash, and everything else about these two runs is identical.
meta = run_meta({"method": "patchcore_ref", "split": "test", "preprocess": PREPROCESS})

for cat in MVTEC_AD_CATEGORIES:
    backend = PatchCoreBackend(seed=SEED, preprocess=PREPROCESS)
    method = PatchCoreRef(backend=backend)
    # Stamped from the method, never from the constant above: the value recorded has to be the
    # one that was applied. This is the seed defect — a number in provenance that nothing
    # applied — kept from walking back in through the other half of the configuration.
    run_evaluation(
        dataset, method, store, {**meta, "preprocess": method.preprocess},
        categories=[cat], split="test", fit_split="train",
        maps_dir=f"{REPRO_CLASSIC}/maps",
    )
    print(f"{cat}: done (seed {method.seed}, preprocess {method.preprocess})")


### 3b.2 Score it, and compare the two reports side by side

Written to its own file. `patchcore_mvtec_ad.md` is the phase-3 record and is not overwritten.

In [ ]:
!python scripts/reproduction_gate.py \
    --results results/reproduction/mvtec_ad_classic/shards \
    --targets configs/reproduction/patchcore_ref.yaml \
    --out results/reproduction/patchcore_mvtec_ad_classic.md

print(open("results/reproduction/patchcore_mvtec_ad_classic.md").read())


In [ ]:
# The one comparison that answers the question this phase was opened for.
import re

def mean_of(path):
    text = open(path).read()
    m = re.search(r"Measured mean I-AUROC \*\*([\d.]+)", text)
    tooth = re.search(r"\| toothbrush \| ([\d.]+)", text)
    return float(m.group(1)), float(tooth.group(1))

anomalib_mean, anomalib_tooth = mean_of("results/reproduction/patchcore_mvtec_ad.md")
classic_mean, classic_tooth = mean_of("results/reproduction/patchcore_mvtec_ad_classic.md")

print(f"{'':10s}  {'mean':>8s}  {'toothbrush':>11s}")
print(f"{'anomalib':10s}  {anomalib_mean:8.2f}  {anomalib_tooth:11.2f}")
print(f"{'classic':10s}  {classic_mean:8.2f}  {classic_tooth:11.2f}")
print(f"{'delta':10s}  {classic_mean - anomalib_mean:+8.2f}  "
      f"{classic_tooth - anomalib_tooth:+11.2f}")
print(f"\npublished: mean 99.0, toothbrush 99.7")


### 3b.3 Three seeds — owed before ANY of this is reported

Protocol §6: three seeds where any stochasticity exists, reported as mean ± std. PatchCore is
stochastic (probe stage 12 measured it), so **neither** report above is reportable on its own —
whichever pre-processing wins, the winning configuration needs seeds 1 and 2 before its number
goes in a table.

Run this for the configuration that phase 3b.2 favours. It writes into the same root: each seed
gets its own shard (`…__seed1.parquet`) and its own map directory, so nothing collides, and
re-running skips whatever already finished.

`--all-seeds` then scores each seed separately and reports the mean of the per-seed means ±
their sample std. The std is reported and never gates — the pre-registered criterion in §2 is on
the mean (§6 v0.2.13). The gate refuses a root that does not hold exactly the three seeds
`configs/reproduction/patchcore_ref.yaml` pre-registers.

In [ ]:
# Set these two to whichever configuration 3b.2 favours, then run.
PREPROCESS = "classic"          # or "anomalib"
ROOT = "results/reproduction/mvtec_ad_classic"

for seed in (1, 2):
    store = ResultStore(f"{ROOT}/shards")
    meta = run_meta({"method": "patchcore_ref", "split": "test", "preprocess": PREPROCESS})
    for cat in MVTEC_AD_CATEGORIES:
        method = PatchCoreRef(backend=PatchCoreBackend(seed=seed, preprocess=PREPROCESS))
        run_evaluation(
            dataset, method, store, {**meta, "preprocess": method.preprocess},
            categories=[cat], split="test", fit_split="train",
            maps_dir=f"{ROOT}/maps",
        )
        print(f"seed {seed} / {cat}: done")


In [ ]:
!python scripts/reproduction_gate.py \
    --results {ROOT}/shards \
    --targets configs/reproduction/patchcore_ref.yaml \
    --all-seeds \
    --out results/reproduction/patchcore_mvtec_ad_3seed.md

print(open("results/reproduction/patchcore_mvtec_ad_3seed.md").read())


## Phase 4 — freeze provenance, once the gate has passed

Only after a PASS. The config records the version **the gate passed on**, so writing it
earlier would record a version whose gate never ran.

1. Put the `ANOMALIB_VERSION` from cell 0.2 into `configs/methods/patchcore_ref.yaml`,
   replacing the `">=1.1"` range.
2. Commit both reproduction reports — they are the evidence for protocol §2.
3. Push, and confirm both CI jobs (3.11, 3.13) stay green. CI never imports anomalib; what it
   exercises from this work is that `patchcore_backend.py` imports lazily.

### Then: the first real threshold calibration

PatchCore is the first *real* method that can produce a calibration artifact
(`intensity_baseline` is the floor, not a detector). Note the **separate results root** — a
validation run and a test run written to the same root collide, and the CLI refuses any split
but `validation`, so a mistake raises rather than silently calibrating on test data.

```bash
python scripts/run_eval.py --method patchcore_ref --root data/mvtec_ad2 \
    --results results/patchcore/val --maps-dir results/patchcore/val/maps \
    --split validation --category vial --seed 0

python scripts/calibrate_threshold.py --results results/patchcore/val \
    --dataset mvtec_ad2 --method patchcore_ref \
    --out configs/thresholds/mvtec_ad2__patchcore_ref.yaml
```

Commit that YAML — committing it before any submission is what makes the pre-registration
auditable.